# 05 · Van der Pol Oscillator — Recovering Nonlinear Damping from Vibration Data

## The Engineering / Biomedical Problem

The Van der Pol (VdP) oscillator is a canonical model of **self-sustained nonlinear oscillation** — systems that actively draw energy to maintain a stable oscillation amplitude. It appears in:

| Application | Physical meaning of $\mu$ | Why estimate $\mu$? |
|-------------|--------------------------|---------------------|
| Cardiac pacemaker cells | Nonlinear ion channel conductance | Assess arrhythmia risk; monitor drug effects on electrophysiology |
| Neuronal firing (FitzHugh–Nagumo approx.) | Excitability parameter | Classify neural dynamics from extracellular recording |
| Chemical oscillations (Belousov–Zhabotinsky) | Nonlinear kinetics | Characterise oscillatory reaction systems |
| Pressure oscillations in packed-bed reactors | Nonlinear flow resistance | Detect channelling, fouling, or mechanical failure |
| Respiratory mechanics | Nonlinear airway resistance | Pulmonary function testing from flow-volume loops |

**The inverse problem:** given a noisy time-series of the oscillation $x(t)$, estimate $\mu$ — the nonlinear damping coefficient.

## The Model

$$\ddot{x} - \mu\,(1-x^2)\,\dot{x} + x = 0$$

When $|x| < 1$: $(1-x^2) > 0$, damping is negative (system adds energy). When $|x| > 1$: $(1-x^2) < 0$, damping is positive (system removes energy). This self-regulating mechanism produces a **stable limit cycle** — a closed orbit in phase space that the trajectory converges to regardless of initial conditions.

| $\mu$ value | Dynamics | PINN challenge |
|---|---|---|
| $\mu \to 0$ | Nearly sinusoidal, weak nonlinearity | Easy |
| $\mu = 1.0$ | Moderate distortion, one limit cycle | Tractable |
| $\mu = 5.0$ | Sharp "relaxation oscillations", high-frequency spikes | **Spectral bias failure** |

## The Reference Paper

**van der Pol, B. (1926).** On relaxation-oscillations. *Phil. Mag.* 2(11):978–992 — origin of the equation.

For PINN context: this inverse problem appears as a benchmark in **Wang, S., Teng, Y. & Perdikaris, P. (2021)** (gradient pathologies in PINNs) and in multiple PINN oscillator estimation papers. The $\mu = 5$ failure mode is the core motivation for Fourier feature inputs and causal training strategies.

## Why PINN Instead of Classical Approaches?

### Classical fitting:
1. **Harmonic balance** — only works for weakly nonlinear systems ($\mu \ll 1$)
2. **Numerical differentiation + regression** — differentiate $x(t)$ twice (numerically), then regress $\ddot{x}$ on $(x, \dot{x}, x^2\dot{x})$. Noise amplification in numerical differentiation makes this unstable.
3. **Extended Kalman Filter (EKF)** — handles noisy nonlinear systems; estimates $\mu$ as an augmented state. Works but requires specifying process noise covariance.
4. **Nonlinear optimisation + ODE solver** — standard approach; sensitive to initial guess for $\mu$.

### PINN approach — the time-scaling trick:
The key innovation for second-order systems: **scale time to $\tau = t/T \in [0,1]$ and transform derivatives back before computing the residual.**

Without scaling, $\dot{x}$ is $T \times$ too large and $\ddot{x}$ is $T^2 \times$ too large in scaled time. For $T = 16$ s, this means residual terms differ by factors of 16 and 256 — the loss is dominated by the largest-magnitude term and training collapses.

With scaling: convert back to real-time derivatives inside the loss function (`xt/T, xtt/(T*T)`). All residual terms are $O(1)$.

This is a universal principle for oscillatory PINNs: **always normalise residual terms to comparable magnitude before training.**

## What the Code Does Step by Step

1. **Reference trajectory** — generated with `scipy.integrate.solve_ivp` at $\mu_{true} = 1.0$, $T = 16$ s, 2.4 oscillation periods
2. **Measurements** — 45 sparse noisy points from the trajectory (5% noise)
3. **Network** — 1 input $\tau \in [0,1]$ → 3 Tanh hidden layers (64 units) → 1 output $x$
4. **Learnable parameter** — `raw_mu` via softplus (always positive)
5. **Nested autograd** — `xt = grad(x, tau)`, then `xtt = grad(xt, tau)` — two passes
6. **Residual** — `xtt/(T²) - mu*(1-x²)*xt/T + x` (all O(1) after scaling)
7. **Training** — Adam (20,000 epochs, lr=3e-3) → L-BFGS (600 steps)
8. **Results** — $\hat{\mu} = 1.075$ (true 1.0): 7.5% error — excellent given 5% noise and strong nonlinearity

## The Spectral Bias Problem (Exercise)

Set `mu_true = 5.0` and re-run. The PINN will fail: the loss stagnates and the predicted trajectory looks like a smooth sinusoid instead of the sharp relaxation oscillations. 

This is **spectral bias** — neural networks learn low-frequency components first and are exponentially slow to learn high-frequency content. For $\mu = 5$, the relaxation oscillations have significant power at $2\omega_0$, $3\omega_0$, ... which a plain MLP cannot capture.

**Fix:** Add Fourier feature inputs: embed $\tau$ as $[\sin(2\pi k \tau / T), \cos(2\pi k \tau / T)]$ for $k = 1, ..., N_f$ before the MLP. This is implemented in Module 01, Notebook 12 (Architectures and Training).


In [1]:
import numpy as np, torch, torch.nn as nn
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

mu_true = 1.0
def vdp(t,y): return [y[1], mu_true*(1-y[0]**2)*y[1] - y[0]]
T = 16.0                                    # ~2.4 periods
sol = solve_ivp(vdp, [0,T], [2.0,0.0], t_eval=np.linspace(0,T,400), rtol=1e-9, atol=1e-9)
tg, xg = sol.t, sol.y[0]

# sparse noisy measurements
idx = np.linspace(0, len(tg)-1, 45).astype(int)
t_meas = tg[idx]; x_meas = xg[idx] + 0.05*np.random.randn(len(idx))
t_data = torch.tensor((t_meas/T).reshape(-1,1), dtype=torch.float32, device=device)
x_data = torch.tensor(x_meas.reshape(-1,1), dtype=torch.float32, device=device)
print("measurements:", len(idx))


Using device: cpu
measurements: 45


In [ ]:
class Net(nn.Module):
    def __init__(self,h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1,h),nn.Tanh(),
                                 nn.Linear(h,h),nn.Tanh(),
                                 nn.Linear(h,h),nn.Tanh(),
                                 nn.Linear(h,1))
        for m in self.net:
            if isinstance(m,nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,t): return self.net(t)

model = Net().to(device)
raw_mu = nn.Parameter(torch.tensor(0.1, device=device))   # learnable mu (via softplus)
sp = torch.nn.functional.softplus

tau = torch.linspace(0,1,300,device=device).reshape(-1,1).requires_grad_(True)
tau0 = torch.zeros(1,1,device=device,requires_grad=True)

def derivs(t):
    x = model(t)
    xt = torch.autograd.grad(x, t, torch.ones_like(x), create_graph=True)[0]
    xtt = torch.autograd.grad(xt, t, torch.ones_like(xt), create_graph=True)[0]
    return x, xt/T, xtt/(T*T)          # convert scaled-time derivs to real time

def loss_fn():
    x, xdot, xddot = derivs(tau)
    mu = sp(raw_mu)
    res = xddot - mu*(1-x**2)*xdot + x            # all O(1)
    l_phys = torch.mean(res**2)
    l_data = torch.mean((model(t_data)-x_data)**2)
    x0,xd0,_ = derivs(tau0)
    l_ic = (x0-2.0)**2 + (xd0-0.0)**2
    return l_data + 0.1*l_phys + l_ic.squeeze(), l_data, l_phys


In [ ]:
allp = list(model.parameters()) + [raw_mu]
opt = torch.optim.Adam(allp, lr=3e-3)
EPOCHS = 20000
for ep in range(EPOCHS):
    opt.zero_grad(); L,ld,lp = loss_fn(); L.backward(); opt.step()
    if ep % 5000 == 0:
        print(f"ep {ep:5d} | data {ld.item():.2e} phys {lp.item():.2e} | mu={sp(raw_mu).item():.3f}")

opt2 = torch.optim.LBFGS(allp, max_iter=600, line_search_fn="strong_wolfe")
def closure():
    opt2.zero_grad(); L,_,_ = loss_fn(); L.backward(); return L
opt2.step(closure)
print(f"\nRecovered:  mu = {sp(raw_mu).item():.3f}   (true {mu_true})")


In [ ]:
tt = torch.linspace(0,1,400,device=device).reshape(-1,1)
with torch.no_grad(): xp = model(tt).cpu().numpy().ravel()
plt.figure(figsize=(9,4.5))
plt.plot(tg,xg,"k--",alpha=.5,label="true trajectory")
plt.plot(tt.cpu().numpy().ravel()*T, xp,"b-",lw=2,label="PINN")
plt.scatter(t_meas,x_meas,c="r",s=18,zorder=5,label="noisy data")
plt.xlabel("time"); plt.ylabel("x(t)")
plt.title(f"Van der Pol oscillator — recovered mu={sp(raw_mu).item():.3f}")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()


---
## 🔬⚡🧠 Three-Method Comparison: Van der Pol Oscillator
**Pure PINN** (uniform) vs **RAR** (adaptive) vs **DeepONet** (operator learning).

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt, time
from scipy.integrate import solve_ivp
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sp = torch.nn.functional.softplus

def make_net(layers):
    seq = []
    for i in range(len(layers)-1):
        seq.append(nn.Linear(layers[i], layers[i+1]))
        if i < len(layers)-2: seq.append(nn.Tanh())
    net = nn.Sequential(*seq)
    for m in net:
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    return net.to(device)

def grad(y,x): return torch.autograd.grad(y,x,torch.ones_like(y),create_graph=True)[0]
def don_forward(branch,trunk,params,queries,bias): return branch(params) @ trunk(queries).T + bias

from scipy.integrate import solve_ivp
mu_true=1.0; T_vdp=16.0
def vdp(t,y): return [y[1], mu_true*(1-y[0]**2)*y[1]-y[0]]
sol=solve_ivp(vdp,[0,T_vdp],[2.,0.],t_eval=np.linspace(0,T_vdp,400),rtol=1e-9,atol=1e-9)
tg_v,xg=sol.t,sol.y[0]
idx=np.linspace(0,len(tg_v)-1,45).astype(int)
t_meas_v=tg_v[idx]; x_meas_v=xg[idx]+0.05*np.random.randn(45)
t_data_v=torch.tensor((t_meas_v/T_vdp).reshape(-1,1),dtype=torch.float32,device=device)
x_data_v=torch.tensor(x_meas_v.reshape(-1,1),dtype=torch.float32,device=device)
tau0_v=torch.zeros(1,1,device=device,requires_grad=True)
ic_v=torch.tensor([[2.,0.]],device=device)

def derivs_v(model,t):
    x=model(t); xt=grad(x,t)/T_vdp; xtt=grad(xt,t)/T_vdp; return x,xt,xtt

# ─── Method 1: Pure PINN ─────────────────────────────────────────────────────
model_vdp=make_net([1,64,64,64,1]); raw_mu=nn.Parameter(torch.tensor(0.1,device=device))
opt_v=torch.optim.Adam(list(model_vdp.parameters())+[raw_mu],lr=3e-3)
tau_v=torch.linspace(0,1,300,device=device).view(-1,1).requires_grad_(True)
def loss_vdp(model,tc):
    x,xd,xdd=derivs_v(model,tc); mu=sp(raw_mu)
    res=xdd-mu*(1-x**2)*xd+x
    x0,xd0,_=derivs_v(model,tau0_v)
    return torch.mean(res**2), torch.mean((model(t_data_v)-x_data_v)**2), (x0-2.)**2+(xd0-0.)**2

t0=time.time()
for ep in range(20000):
    opt_v.zero_grad(); lp,ld,li=loss_vdp(model_vdp,tau_v); L=ld+0.1*lp+li.squeeze(); L.backward(); opt_v.step()
    if ep%5000==0: print(f"  [PINN VdP] ep{ep:5d} mu={sp(raw_mu).item():.3f}")
opt_v2=torch.optim.LBFGS(list(model_vdp.parameters())+[raw_mu],max_iter=400,line_search_fn="strong_wolfe")
def clos_v():
    opt_v2.zero_grad(); lp,ld,li=loss_vdp(model_vdp,tau_v); (ld+0.1*lp+li.squeeze()).backward(); return ld
opt_v2.step(clos_v); time_pinn=time.time()-t0
model_vdp.eval(); tt_v=torch.linspace(0,1,400,device=device).view(-1,1)
with torch.no_grad(): xp_pinn=model_vdp(tt_v).cpu().numpy().ravel()
xg_i=np.interp(tt_v.cpu().numpy().ravel()*T_vdp,tg_v,xg)
err_pinn_v=np.sqrt(np.mean((xp_pinn-xg_i)**2))/np.sqrt(np.mean(xg_i**2))
print(f"  PINN VdP: mu={sp(raw_mu).item():.3f}(true 1.0) L2={err_pinn_v:.4f} t={time_pinn:.0f}s")

# ─── Method 2: RAR ───────────────────────────────────────────────────────────
model_rar_v=make_net([1,64,64,64,1]); raw_mu_r=nn.Parameter(torch.tensor(0.1,device=device))
opt_rv=torch.optim.Adam(list(model_rar_v.parameters())+[raw_mu_r],lr=3e-3)
tau_rar_v=torch.linspace(0,1,50,device=device).view(-1,1).requires_grad_(True)
def loss_rar_v(model,tc):
    x,xd,xdd=derivs_v(model,tc); mu=sp(raw_mu_r)
    res=xdd-mu*(1-x**2)*xd+x
    x0,xd0,_=derivs_v(model,tau0_v)
    return torch.mean(res**2), torch.mean((model(t_data_v)-x_data_v)**2), (x0-2.)**2+(xd0-0.)**2

t0=time.time()
for rnd in range(8):
    for ep in range(2500):
        opt_rv.zero_grad(); lp,ld,li=loss_rar_v(model_rar_v,tau_rar_v); L=ld+0.1*lp+li.squeeze(); L.backward(); opt_rv.step()
    cand=torch.rand(5000,1,device=device).requires_grad_(True)
    x_c,xd_c,xdd_c=derivs_v(model_rar_v,cand); mu_r=sp(raw_mu_r)
    res_c=(xdd_c-mu_r*(1-x_c**2)*xd_c+x_c).abs().detach().cpu().numpy().ravel()
    idx=np.argsort(res_c)[-25:]; new_pts=cand[idx].detach()
    tau_rar_v=torch.cat([tau_rar_v.detach(),new_pts]).requires_grad_(True)
    opt_rv=torch.optim.Adam(list(model_rar_v.parameters())+[raw_mu_r],lr=1e-3)
    print(f"  [RAR VdP] round {rnd+1}: {tau_rar_v.shape[0]} pts mu={sp(raw_mu_r).item():.3f}")
time_rar=time.time()-t0
model_rar_v.eval()
with torch.no_grad(): xp_rar=model_rar_v(tt_v).cpu().numpy().ravel()
err_rar_v=np.sqrt(np.mean((xp_rar-xg_i)**2))/np.sqrt(np.mean(xg_i**2))
print(f"  RAR VdP: mu={sp(raw_mu_r).item():.3f} L2={err_rar_v:.4f} t={time_rar:.0f}s pts={tau_rar_v.shape[0]}")

# ─── Method 3: DeepONet ──────────────────────────────────────────────────────
p=64; branch_v=make_net([1,64,64,p]); trunk_v=make_net([1,64,64,p])
bias_v=nn.Parameter(torch.zeros(1,device=device))
opt_dv=torch.optim.Adam(list(branch_v.parameters())+list(trunk_v.parameters())+[bias_v],lr=3e-3)
N_tr=1000; N_q=50; mu_samples=np.random.uniform(0.1,3.0,N_tr); tq_v=np.random.rand(N_q)*T_vdp
def vdp_sol(mu,t_arr):
    def f(t,y): return [y[1],mu*(1-y[0]**2)*y[1]-y[0]]
    try: sol=solve_ivp(f,[0,T_vdp],[2.,0.],t_eval=t_arr,rtol=1e-7,atol=1e-9); return sol.y[0]/3.
    except: return np.zeros(len(t_arr))
print("  Generating VdP DeepONet data...")
Y_vdp_d=np.array([vdp_sol(mu_samples[i],tq_v) for i in range(N_tr)])
params_dv=torch.tensor(mu_samples.reshape(-1,1)/3.,dtype=torch.float32,device=device)
tq_vt=torch.tensor((tq_v/T_vdp).reshape(-1,1),dtype=torch.float32,device=device)
Y_dv=torch.tensor(Y_vdp_d,dtype=torch.float32,device=device)
t0=time.time()
for ep in range(10000):
    opt_dv.zero_grad(); pred=don_forward(branch_v,trunk_v,params_dv,tq_vt,bias_v)
    L=torch.mean((pred-Y_dv)**2); L.backward(); opt_dv.step()
    if ep%2000==0: print(f"  [DeepONet VdP] ep {ep} loss {L.item():.3e}")
time_don=time.time()-t0
branch_v.eval(); trunk_v.eval()
param_test_v=torch.tensor([[mu_true/3.]],dtype=torch.float32,device=device)
t_don_v=torch.tensor((np.linspace(0,T_vdp,400)/T_vdp).reshape(-1,1),dtype=torch.float32,device=device)
with torch.no_grad(): x_don=don_forward(branch_v,trunk_v,param_test_v,t_don_v,bias_v).squeeze().cpu().numpy()*3.
err_don_v=np.sqrt(np.mean((x_don-xg_i)**2))/np.sqrt(np.mean(xg_i**2))
print(f"  DeepONet VdP: mu=1.0 L2={err_don_v:.4f} t={time_don:.0f}s")

# ─── Comparison ──────────────────────────────────────────────────────────────
print("\n"+"="*55+"\nMETHOD COMPARISON — Van der Pol Oscillator\n"+"="*55)
print(f"{'Method':<25}{'Rel L2':>9}{'mu_est':>8}{'Time(s)':>9}")
print(f"{'Pure PINN':<25}{err_pinn_v:>9.4f}{sp(raw_mu).item():>8.3f}{time_pinn:>9.0f}")
print(f"{'RAR':<25}{err_rar_v:>9.4f}{sp(raw_mu_r).item():>8.3f}{time_rar:>9.0f}")
print(f"{'DeepONet':<25}{err_don_v:>9.4f}{'N/A':>8}{time_don:>9.0f}")
tg_v_plot=tt_v.cpu().numpy().ravel()*T_vdp
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot(tg_v,xg,'k-',lw=2.5,label='True'); axes[0].scatter(t_meas_v,x_meas_v,c='r',s=15,zorder=5,label='Noisy data')
axes[0].plot(tg_v_plot,xp_pinn,'b--',lw=2,label=f'PINN (mu={sp(raw_mu).item():.2f})')
axes[0].plot(tg_v_plot,xp_rar,'r-.',lw=2,label=f'RAR (mu={sp(raw_mu_r).item():.2f})')
axes[0].plot(tg_v_plot,x_don,'g:',lw=2,label='DeepONet')
axes[0].set_xlabel('time'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3); axes[0].set_title('Van der Pol')
axes[1].bar(['Pure PINN','RAR','DeepONet'],[err_pinn_v,err_rar_v,err_don_v],color=['steelblue','firebrick','seagreen'])
axes[1].set_ylabel('Rel L2'); axes[1].set_title('Accuracy'); axes[1].grid(axis='y',alpha=0.3)
pt_vr=tau_rar_v.detach().cpu().numpy().ravel()*T_vdp
axes[2].hist(pt_vr,bins=25,color='firebrick',alpha=0.7,label='RAR'); axes[2].hist(np.linspace(0,T_vdp,300),bins=25,color='steelblue',alpha=0.5,label='Uniform')
axes[2].set_xlabel('time'); axes[2].set_title('Collocation Density'); axes[2].legend()
plt.tight_layout(); plt.savefig('comparison_vdp.png',dpi=120,bbox_inches='tight'); plt.show()
